# Improving BRMA on TON_IoT DDoS (one run, about 3 hours, CPU, no internet)
Tests three variants of the memory audit against the original BRMA on the dataset where it was weakest:

* **BRMA-M**: the trained model's attack-vs-benign score with the same conformal 5% threshold.
* **BRMA-G**: normal traffic modelled as a mixture of 5 Gaussians instead of one.
* **BRMA-C**: flag a record if either the original BRMA test or the model test fires (each at 2.5%, so the 5% guarantee holds).

**Phase 1 (selection):** development seeds 101-103, dev split, fixed rule picks one variant.
**Phase 2 (evaluation):** 12 new seeds (201-212), test split, pre-specified tests T11-T14.

**How to run:** Add Input -> attach the same TON_IoT dataset (TON_IoT_Train_Test_Network.csv). Accelerator **None (CPU)**.
**Save Version -> Save & Run All (Commit)**. When finished, download `ton-improve.zip` from the Output tab and send it.

In [ ]:
# 1. Locate and check train_test_network.csv in the attached inputs
import os, sys, glob, json, subprocess, time, hashlib
import pandas as pd
W = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
cands = [p for p in glob.glob("/kaggle/input/**/*.csv", recursive=True)]
named = [p for p in cands if "train_test_network" in os.path.basename(p).lower().replace("-", "_").replace(" ", "_")]
CSV = None
for p in named + [c for c in cands if c not in named]:
    try:
        cols = [c.strip().lower() for c in pd.read_csv(p, nrows=5).columns]
    except Exception:
        continue
    if "type" in cols and "label" in cols and "src_bytes" in cols:
        CSV = p; break
if CSV is None:
    raise SystemExit("No TON_IoT network CSV found. Attach a dataset with train_test_network.csv (Add Input -> search TON_IoT).")
n = sum(1 for _ in open(CSV)) - 1
print("using:", CSV, "| rows", n)
if not 400_000 <= n <= 500_000:
    print("WARNING: expected about 461,043 rows (official train_test_network.csv).")

In [ ]:
# 2. Write the study code (ton_study.py unchanged + ton_improve.py)
TON_SRC = '"""BRMA on TON_IoT (extension study): Benign-Reference Memory Audit for federated class-incremental IDS.\n\nSelf-contained re-implementation of the project\'s audited Phase 3 learner (src/phase3 + src/phase3_v2)\nfor a second dataset, generalised to any feature width and class count:\n  * MLP d-128-64-width (dropout .1), head expanded when new classes arrive (same seeded initialisation);\n  * 5 clients, FedAvg weighted by admitted rows, 12 rounds x 1 epoch per stage, Adam lr 1e-3 reset each broadcast;\n  * DR learner = current CE + 1.0 replay CE + 0.1 dark-logit distillation, 500-record reservoir per client;\n  * FT (no replay), EI (empirical-Fisher EWC, lambda 100), DC/DG low-loss admission filters, MEDIAN aggregation;\n  * k-client label flip of the stage-2 target class (DDoS) to benign; BRMA, RAND, LOSS, ORACLE memory actions.\nBRMA is identical to the CIC-IDS2017 study (tied-covariance likelihood ratio, Ledoit-Wolf, conformal alpha .05).\n\nUsage:\n  python ton_study.py prepare --csv train_test_network.csv --data DATA_DIR\n  python ton_study.py run --data DATA_DIR --out OUT --seeds 1 2 3 4 --workers 4 [--arms C K3H] [--full MEDIAN DC]\n  python ton_study.py summarize --data DATA_DIR --out OUT\n"""\nfrom __future__ import annotations\nimport os\nfor _k in (\'OMP_NUM_THREADS\', \'MKL_NUM_THREADS\', \'OPENBLAS_NUM_THREADS\', \'NUMEXPR_NUM_THREADS\'):\n    os.environ[_k] = \'1\'\nimport argparse, copy, json, sys, time, traceback\nimport concurrent.futures as futures\nfrom dataclasses import dataclass, asdict\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n# ---------------------------------------------------------------- dataset protocol (fixed before any run)\nCLASSES = [\'normal\', \'scanning\', \'dos\', \'ddos\', \'password\', \'backdoor\', \'injection\', \'xss\', \'ransomware\', \'mitm\']\nARRIVAL = [1, 1, 1, 2, 2, 2, 3, 3, 4, 4]          # stage in which each class first appears (normal: every stage)\nK, BENIGN = len(CLASSES), 0\nTARGET = CLASSES.index(\'ddos\')                    # flipped class (plays the role of DoS Hulk in CIC-IDS2017)\nCAPACITY, BATCH = 500, 128\nSEEDS = tuple(range(1, 12)) + (42,)\nARMS = {\'C\': (0, 0, 1), \'K1\': (1, 1, 2), \'K3\': (3, 1, 2), \'K3H\': (3, 9, 10), \'K5\': (5, 1, 2)}\nBRANCH_METHODS = (\'NONE\', \'BRMA\', \'RAND\', \'LOSS\', \'ORACLE\')\nFULL_METHODS = (\'FT\', \'EI\', \'DC\', \'DG\', \'MEDIAN\')\nALPHA = 0.05\nFIT_CAP, POOL_CAP, MIN_ROWS = 50000, 200000, 20\nDIRICHLET = 1.0                                   # non-IID client allocation per class\nCARILLO_F1_ALL = 74.49                            # Carillo et al., IEEE TNSM 2026, Table IV (TON_IoT, FedProx+MEMENTO+)\n\n\n@dataclass(frozen=True)\nclass Settings:\n    learner: str = \'DR\'\n    admission: str = \'NONE\'\n    seed: int = 1\n    rounds: int = int(os.environ.get(\'TON_ROUNDS\', \'12\'))\n    epochs: int = 1\n    learning_rate: float = .001\n    lambda_E: float = 100.\n    lambda_y: float = 1.\n    lambda_z: float = .1\n    aggregation: str = \'fedavg\'\n    dose: tuple = (0, 1)\n    capacity: int = 500          # replay records per client\n    hidden: tuple = (128, 64)    # MLP widths\n    class_weight: bool = False   # client-local inverse-sqrt-frequency weights on the current-data CE\n\n\ndef rng(namespace, seed, *scope):\n    return np.random.Generator(np.random.PCG64(np.random.SeedSequence([7, namespace, seed, *scope])))\n\n\ndef torch_seed(namespace, seed, *scope):\n    return int(rng(namespace, seed, *scope).integers(0, 2**63, dtype=np.int64))\n\n\ndef setup():\n    torch.set_num_threads(1)\n    try:\n        torch.set_num_interop_threads(1)\n    except RuntimeError:\n        pass\n    torch.use_deterministic_algorithms(True)\n    torch.set_float32_matmul_precision(\'highest\')\n\n\n# ---------------------------------------------------------------- preparation (train/dev/test, features)\ndef prepare(csv, folder, ports=False):\n    import pandas as pd\n    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)\n    df = pd.read_csv(csv, low_memory=False)\n    df.columns = [str(c).strip().lower() for c in df.columns]\n    if \'type\' not in df.columns:\n        raise ValueError(f\'No "type" column; columns are {list(df.columns)}\')\n    t = df[\'type\'].astype(str).str.strip().str.lower()\n    unknown = sorted(set(t) - set(CLASSES))\n    if unknown:\n        raise ValueError(f\'Unexpected classes {unknown}\')\n    y = t.map({c: i for i, c in enumerate(CLASSES)}).to_numpy(np.int64)\n    drop = {\'ts\', \'src_ip\', \'dst_ip\', \'src_port\', \'dst_port\', \'label\', \'type\'}\n    feats, names, dropped = [], [], []\n    for c in df.columns:\n        if c in drop:\n            continue\n        s = df[c]\n        if s.dtype == bool or pd.api.types.is_numeric_dtype(s):\n            v = pd.to_numeric(s, errors=\'coerce\').fillna(0).to_numpy(np.float64)\n            feats.append(np.sign(v) * np.log1p(np.abs(v))); names.append(c); continue\n        ss = s.astype(str).str.strip()\n        placeholder = ss.isin([\'-\', \'\', \'nan\'])\n        num = pd.to_numeric(ss.where(~placeholder), errors=\'coerce\')\n        nunique = ss.nunique()\n        if (~placeholder).sum() and num[~placeholder].notna().mean() >= 0.99 and nunique > 50:\n            v = num.fillna(0).to_numpy(np.float64); feats.append(np.sign(v) * np.log1p(np.abs(v))); names.append(c)\n        elif nunique <= 50:\n            for val in sorted(ss.unique()):\n                feats.append((ss == val).to_numpy(np.float64)); names.append(f\'{c}={val}\')\n        else:\n            dropped.append(c)\n    X = np.column_stack(feats).astype(np.float32)\n    n_raw = len(y)\n    frame = pd.DataFrame(X); frame[\'__y\'] = y\n    keep = ~frame.duplicated().to_numpy()            # de-duplication always on the base features: same rows and splits in every variant\n    if ports:                                        # destination-port features: signed log1p value + one-hot of the 30 most frequent ports\n        p = pd.to_numeric(df[\'dst_port\'], errors=\'coerce\').fillna(-1).astype(np.int64).to_numpy()\n        top = pd.Series(p[keep]).value_counts().index[:30]\n        extra = [np.log1p(np.maximum(p, 0)).astype(np.float64)] + [(p == v).astype(np.float64) for v in top]\n        X = np.column_stack([X] + extra).astype(np.float32); names += [\'dst_port\'] + [f\'dst_port={v}\' for v in top]\n    X, y = X[keep], y[keep]\n    conflict = int(pd.DataFrame(X).duplicated(keep=False).sum())\n    split = np.zeros(len(y), np.int8); stage = np.zeros(len(y), np.int8)   # split 0 train, 1 dev, 2 test\n    for c in range(K):\n        idx = rng(1, 0, c).permutation(np.flatnonzero(y == c))\n        ntr, ndv = int(.7 * len(idx)), int(.15 * len(idx))\n        split[idx[ntr:ntr + ndv]] = 1; split[idx[ntr + ndv:]] = 2\n        tr = idx[:ntr]\n        if c == BENIGN:\n            for s, part in enumerate(np.array_split(rng(2, 0).permutation(tr), 4)):\n                stage[part] = s + 1\n        else:\n            stage[tr] = ARRIVAL[c]\n    tr = np.flatnonzero(split == 0)\n    mu, sd = X[tr].mean(0), X[tr].std(0)\n    sd[sd < 1e-6] = 1.\n    X = np.clip((X - mu) / sd, -10, 10).astype(np.float32)\n    order = tr[rng(3, 0).permutation(len(tr))]                # global record order -> ids 1..n\n    dv, te = np.flatnonzero(split == 1), np.flatnonzero(split == 2)\n    np.savez(folder / \'ton.npz\', xtr=X[order], ytr=y[order], stage=stage[order], ids=np.arange(1, len(order) + 1, dtype=np.uint64),\n             xdev=X[dv], ydev=y[dv], xte=X[te], yte=y[te], names=np.asarray(names))\n    counts = {CLASSES[c]: {sp: int(((y == c) & (split == i)).sum()) for i, sp in enumerate((\'train\', \'dev\', \'test\'))} for c in range(K)}\n    report = dict(csv=str(csv), raw_rows=n_raw, duplicate_rows_removed=int(n_raw - len(y)), rows=int(len(y)),\n                  feature_vectors_shared_by_two_classes=conflict, features=len(names), dropped_columns=dropped,\n                  arrival={c: ARRIVAL[i] for i, c in enumerate(CLASSES)}, target=CLASSES[TARGET], counts=counts,\n                  train_rows_per_stage={s: int((stage[order] == s).sum()) for s in range(1, 5)},\n                  destination_port_features=bool(ports), scaling=\'signed log1p for numeric columns, one-hot for <=50 categories, z-score with train statistics, clip +-10\')\n    (folder / \'prepare-report.json\').write_text(json.dumps(report, indent=1))\n    return report\n\n\n_DATA = {}\n\n\ndef load(folder):\n    folder = str(folder)\n    if folder not in _DATA:\n        with np.load(Path(folder) / \'ton.npz\', allow_pickle=False) as f:\n            _DATA[folder] = {k: f[k] for k in f.files}\n    return _DATA[folder]\n\n\nclass Stage:\n    def __init__(self, stage, x, y, ids, owner):\n        self.stage, self.x, self.y, self.ids, self.owner = stage, x, y, ids, owner\n\n    def targets(self, slots):\n        lut = np.full(K, -1, np.int64); lut[slots] = np.arange(len(slots)); return lut[self.y]\n\n\ndef ownership(seed, y):\n    owner = np.zeros(len(y), np.int8)\n    for c in range(K):\n        idx = np.flatnonzero(y == c)\n        p = rng(20, seed, c).dirichlet(np.full(5, DIRICHLET))\n        owner[idx] = rng(21, seed, c).choice(5, len(idx), p=p)\n    return owner\n\n\ndef corrupt_multi(clean, ids, owner, seed, stage, k, num, den):\n    """k-client nested label flip of the target class to benign in stage 2 (same streams as the CIC study)."""\n    supplied = clean.copy()\n    info = dict(stage=stage, k=k, dose=[num, den], attackers=[], poison_ids=[])\n    if stage != 2 or k == 0 or num == 0:\n        return supplied, info\n    counts = np.bincount(owner[clean == TARGET], minlength=5)\n    eligible = np.flatnonzero(counts >= 10)\n    first = int(rng(22, seed).choice(eligible))\n    rest = [int(c) for c in rng(23, seed).permutation(eligible) if c != first]\n    attackers = [first] + rest[:k - 1]\n    poison = []\n    for a in attackers:\n        source = np.flatnonzero((owner == a) & (clean == TARGET))\n        source = source[np.argsort(ids[source], kind=\'stable\')]\n        chosen = rng(24, seed, a).permutation(source)[:num * len(source) // den]\n        supplied[chosen] = BENIGN; poison.extend(int(v) for v in ids[chosen])\n    info.update(attackers=attackers, poison_ids=poison)\n    return supplied, info\n\n\ndef inputs(folder, seed, arm):\n    D = load(folder)\n    owner = ownership(seed, D[\'ytr\'])\n    k, num, den = ARMS[arm]\n    data, attack = [], None\n    for s in range(1, 5):\n        idx = np.flatnonzero(D[\'stage\'] == s)\n        y, info = corrupt_multi(D[\'ytr\'][idx], D[\'ids\'][idx], owner[idx], seed, s, k, num, den)\n        if s == 2:\n            attack = info\n        data.append(Stage(s, D[\'xtr\'][idx], y, D[\'ids\'][idx], owner[idx]))\n    return data, attack\n\n\n# ---------------------------------------------------------------- model and learner (mirrors src/phase3 + src/phase3_v2)\nclass MLP(nn.Module):\n    def __init__(self, d, width, seed, hidden=(128, 64)):\n        super().__init__()\n        h1, h2 = hidden; self.h2 = h2\n        with torch.random.fork_rng(devices=[]):\n            torch.manual_seed(torch_seed(4, seed))\n            self.backbone = nn.Sequential(nn.Linear(d, h1), nn.ReLU(), nn.Dropout(.1), nn.Linear(h1, h2), nn.ReLU(), nn.Dropout(.1))\n            self.classifier = nn.Linear(h2, width)\n        self.initialize_rows(0, 1, seed)\n\n    def initialize_rows(self, old, stage, seed):\n        with torch.random.fork_rng(devices=[]), torch.no_grad():\n            torch.manual_seed(torch_seed(5, seed, stage))\n            nn.init.xavier_uniform_(self.classifier.weight[old:], gain=1); self.classifier.bias[old:].zero_()\n\n    def expand(self, width, stage, seed):\n        old = self.classifier\n        with torch.random.fork_rng(devices=[]):\n            new = nn.Linear(self.h2, width)\n        with torch.no_grad():\n            new.weight[:old.out_features].copy_(old.weight); new.bias[:old.out_features].copy_(old.bias)\n        self.classifier = new\n        self.initialize_rows(old.out_features, stage, seed)\n\n    def forward(self, x):\n        return self.classifier(self.backbone(x))\n\n\ndef snapshot(model):\n    return {k: v.detach().clone() for k, v in model.state_dict().items()}\n\n\ndef client_state(s, i, d):\n    g = torch.Generator().manual_seed(torch_seed(10, s.seed, i))\n    replay = None\n    if s.learner == \'DR\':\n        cap = s.capacity\n        replay = dict(x=np.zeros((cap, d), np.float32), y=np.zeros(cap, np.int64), ids=np.zeros(cap, np.uint64),\n                      z=np.zeros((cap, K), np.float32), width=np.zeros(cap, np.uint8), size=0, seen=0, cap=cap,\n                      draw=rng(7, s.seed, i), insert=rng(8, s.seed, i))\n    return dict(accepted=np.empty(0, np.int64), dropout=g.get_state().numpy().copy(), replay=replay, importance={}, anchor={})\n\n\ndef insert_refill(m, x, labels, ids, teachers, width):\n    """Reservoir (Algorithm R) that first fills slots freed by an audit; plain reservoir when nothing was removed."""\n    cap = m.get(\'cap\', CAPACITY)\n    for i in range(len(labels)):\n        m[\'seen\'] += 1; seen = m[\'seen\']\n        if m[\'size\'] < min(cap, seen - 1):\n            slot = m[\'size\']; m[\'size\'] += 1\n        else:\n            slot = seen - 1 if seen <= cap else int(m[\'insert\'].integers(0, seen))\n            m[\'size\'] = min(cap, seen)\n        if slot < cap:\n            m[\'x\'][slot] = x[i]; m[\'y\'][slot] = labels[i]; m[\'ids\'][slot] = ids[i]\n            m[\'z\'][slot].fill(0); m[\'z\'][slot, :width] = teachers[i]; m[\'width\'][slot] = width\n\n\ndef ce_scores(model, x, targets):\n    model.eval(); out = np.empty(len(targets), np.float32)\n    with torch.no_grad():\n        for a in range(0, len(targets), 4096):\n            out[a:a + 4096] = F.cross_entropy(model(torch.from_numpy(x[a:a + 4096])), torch.from_numpy(targets[a:a + 4096]), reduction=\'none\').numpy()\n    return out\n\n\ndef admit(model, d, slots, new_codes, clients, s):\n    targets = d.targets(slots)\n    scores = ce_scores(model, d.x, targets) if (s.admission in (\'DC\', \'DG\') and d.stage > 1) else None\n    for i, c in enumerate(clients):\n        idx = np.flatnonzero(d.owner == i)\n        if scores is None:\n            chosen = idx\n        elif s.admission == \'DG\':\n            chosen = idx[np.lexsort((d.ids[idx], scores[idx]))][:(3 * len(idx) + 2) // 4]\n        else:\n            chosen = []\n            for code in np.unique(d.y[idx]):\n                g = idx[d.y[idx] == code]\n                n = len(g) if (int(code) in new_codes or len(g) < 20) else (3 * len(g) + 3) // 4\n                chosen.extend(g[np.lexsort((d.ids[g], scores[g]))][:n].tolist())\n            chosen = np.asarray(chosen, np.int64)\n        c[\'accepted\'] = np.sort(chosen).astype(np.int64)\n\n\ndef expand_importance(c, model):\n    for name, value in model.named_parameters():\n        prior, anchor = c[\'importance\'].get(name), c[\'anchor\'].get(name)\n        upd, anc = torch.zeros_like(value), value.detach().clone()\n        if prior is not None:\n            region = tuple(slice(0, n) for n in prior.shape); upd[region] = prior; anc[region] = anchor\n        c[\'importance\'][name], c[\'anchor\'][name] = upd, anc\n\n\ndef empirical_importance(model, x, targets, chunk=64):\n    params = {k: v.detach() for k, v in model.named_parameters()}; buffers = dict(model.named_buffers())\n    totals = {k: torch.zeros_like(p) for k, p in params.items()}\n    model.eval()\n\n    def single(p, r, l):\n        return F.cross_entropy(torch.func.functional_call(model, (p, buffers), (r.unsqueeze(0),)), l.unsqueeze(0))\n    grad = torch.func.vmap(torch.func.grad(single), in_dims=(None, 0, 0), randomness=\'error\')\n    for a in range(0, len(targets), chunk):\n        for k, g in grad(params, x[a:a + chunk], targets[a:a + chunk]).items():\n            totals[k].add_(g.square().sum(0))\n    return {k: v / len(targets) for k, v in totals.items()}\n\n\ndef commit_importance(model, clients, d, slots, s):\n    targets = d.targets(slots); shared = snapshot(model)\n    for i, c in enumerate(clients):\n        if len(c[\'accepted\']):\n            sel = rng(9, s.seed, i, d.stage).choice(c[\'accepted\'], min(1024, len(c[\'accepted\'])), replace=False)\n            for k, v in empirical_importance(model, torch.from_numpy(d.x[sel]), torch.from_numpy(targets[sel])).items():\n                c[\'importance\'][k].add_(v)\n        c[\'anchor\'] = {k: v.clone() for k, v in shared.items()}\n\n\ndef dark_loss(pred, teachers, widths):\n    mask = torch.arange(pred.shape[1])[None, :] < widths[:, None]\n    return (torch.where(mask, pred - teachers[:, :pred.shape[1]], 0.).square().sum(1) / widths).mean()\n\n\ndef train_client(broadcast, template, c, d, slots, s, round_id, cid):\n    if not len(c[\'accepted\']):\n        return None\n    local = copy.deepcopy(template); local.load_state_dict(broadcast)\n    opt = torch.optim.Adam(local.parameters(), lr=s.learning_rate, betas=(.9, .999), eps=1e-8, foreach=False)\n    targets = d.targets(slots); lut = np.full(K, -1, np.int64); lut[slots] = np.arange(len(slots))\n    m = c[\'replay\']\n    cw = None\n    if s.class_weight:                                   # client-local, from its own admitted supplied labels\n        n = np.bincount(targets[c[\'accepted\']], minlength=len(slots)).astype(np.float64)\n        w = np.where(n > 0, 1. / np.sqrt(np.maximum(n, 1.)), 0.)\n        w = w / (w[targets[c[\'accepted\']]].mean())       # mean weight over the client\'s rows = 1\n        cw = torch.tensor(w, dtype=torch.float32)\n    with torch.random.fork_rng(devices=[]):\n        torch.set_rng_state(torch.from_numpy(c[\'dropout\'].copy()))\n        mask = np.zeros(len(d.y), bool); mask[c[\'accepted\']] = True\n        raw = np.flatnonzero(d.owner == cid)\n        for epoch in range(1, s.epochs + 1):\n            perm = rng(6, s.seed, cid, d.stage, round_id, epoch).permutation(raw)\n            order = perm[mask[perm]]\n            for a in range(0, len(order), BATCH):\n                cur = order[a:a + BATCH]\n                x, y = torch.from_numpy(d.x[cur]), torch.from_numpy(targets[cur])\n                first = m is not None and round_id == 1 and epoch == 1\n                if first:\n                    local.eval()\n                    with torch.no_grad():\n                        teachers = local(x).numpy().copy()\n                replay = None\n                if m is not None and m[\'size\']:\n                    draw = m[\'draw\'].choice(m[\'size\'], min(BATCH, m[\'size\']), replace=False)\n                    replay = {k: m[k][draw].copy() for k in (\'x\', \'y\', \'z\', \'width\')}\n                opt.zero_grad(set_to_none=True); local.train()\n                total = F.cross_entropy(local(x), y, weight=cw)\n                if replay is not None:\n                    rx = torch.from_numpy(replay[\'x\'])\n                    total = total + s.lambda_y * F.cross_entropy(local(rx), torch.from_numpy(lut[replay[\'y\']]))\n                    local.eval()\n                    total = total + s.lambda_z * dark_loss(local(rx), torch.from_numpy(replay[\'z\']), torch.from_numpy(replay[\'width\'].astype(np.int64)))\n                    local.train()\n                if s.learner == \'EI\':\n                    pen = torch.zeros(())\n                    for name, p in local.named_parameters():\n                        pen = pen + (c[\'importance\'][name] * (p - c[\'anchor\'][name]).square()).sum()\n                    total = total + pen * (s.lambda_E / 2)\n                if not torch.isfinite(total):\n                    raise FloatingPointError(\'non-finite objective\')\n                total.backward(); opt.step()\n                if first:\n                    insert_refill(m, d.x[cur], d.y[cur], d.ids[cur], teachers, len(slots))\n        c[\'dropout\'] = torch.get_rng_state().numpy().copy()\n    return snapshot(local)\n\n\ndef aggregate(broadcast, updates, counts, how):\n    active = [u for u, n in zip(updates, counts) if n]\n    if how == \'median\' and len(active) >= 3:\n        return {k: torch.quantile(torch.stack([u[k] for u in active]), .5, dim=0) for k in broadcast}\n    total = sum(counts); out = {k: torch.zeros_like(v) for k, v in broadcast.items()}\n    for n, u in zip(counts, updates):\n        if n:\n            w = float(np.float32(n / total))\n            for k in out:\n                out[k].add_(u[k] * w)\n    return out\n\n\ndef evaluate(folder, model, slots, stage):\n    D = load(folder); res = dict(stage=stage, slots=list(map(int, slots)))\n    model.eval()\n    for split, xs, ys in ((\'dev\', \'xdev\', \'ydev\'), (\'test\', \'xte\', \'yte\')):\n        x, y = D[xs], D[ys]; keep = np.flatnonzero(np.isin(y, slots)); M = np.zeros((K, K), np.int64)\n        with torch.no_grad():\n            for a in range(0, len(keep), 8192):\n                ix = keep[a:a + 8192]\n                np.add.at(M, (y[ix], np.asarray(slots)[model(torch.from_numpy(x[ix])).argmax(1).numpy()]), 1)\n        res[split] = M.tolist()\n    return res\n\n\ndef run_stages(folder, model, clients, slots, data, s, start, end):\n    stages = []; d0 = data[0].x.shape[1]\n    for st in range(start, end + 1):\n        d = data[st - 1]\n        new = [int(c) for c in np.unique(d.y) if int(c) not in slots]; slots.extend(new)\n        if model is None:\n            model = MLP(d0, len(slots), s.seed, tuple(s.hidden))\n        elif new:\n            model.expand(len(slots), st, s.seed)\n        if s.learner == \'EI\':\n            for c in clients:\n                expand_importance(c, model)\n        admit(model, d, slots, set(new), clients, s)\n        for r in range(1, s.rounds + 1):\n            b = snapshot(model)\n            ups = [train_client(b, model, c, d, slots, s, r, i) for i, c in enumerate(clients)]\n            model.load_state_dict(aggregate(b, ups, [len(c[\'accepted\']) for c in clients], s.aggregation))\n        if s.learner == \'EI\':\n            commit_importance(model, clients, d, slots, s)\n        stages.append(evaluate(folder, model, slots, st))\n    return model, stages\n\n\n# ---------------------------------------------------------------- BRMA (identical to the CIC-IDS2017 study)\ndef maha_sq(Z, mu, P, chunk=65536):\n    out = np.empty(len(Z))\n    for i in range(0, len(Z), chunk):\n        D = Z[i:i + chunk] - mu; out[i:i + chunk] = np.einsum(\'ij,ij->i\', D @ P, D)\n    return out\n\n\ndef conformal(cal, alpha):\n    n = len(cal); return float(np.quantile(cal, min(1., np.ceil((n + 1) * (1 - alpha)) / n), method=\'higher\'))\n\n\nclass Auditor:\n    def __init__(self, data, seed):\n        self.data, self.seed = data, seed\n        x1 = data[0].x; self.mu0, self.sd0 = x1.mean(0), x1.std(0) + 1e-6\n        ben = rng(41, seed).permutation(np.flatnonzero(data[0].y == BENIGN))\n        self.Zref = self.std(x1[ben[:len(ben) // 2]]); self.Zcal = self.std(x1[ben[len(ben) // 2:]]); self.models = {}\n\n    def std(self, X):\n        return (np.asarray(X, np.float64) - self.mu0) / self.sd0\n\n    def model(self, t):\n        if t in self.models:\n            return self.models[t]\n        from sklearn.covariance import LedoitWolf\n        means, centred = {}, [self.Zref - self.Zref.mean(0)]\n        for c in range(1, K):\n            X = np.concatenate([self.data[s].x[self.data[s].y == c] for s in range(t)])\n            if len(X) < MIN_ROWS:\n                continue\n            if len(X) > FIT_CAP:\n                X = X[rng(42, self.seed, t, c).choice(len(X), FIT_CAP, replace=False)]\n            Z = self.std(X); means[c] = Z.mean(0); centred.append(Z - means[c])\n        pooled = np.concatenate(centred)\n        if len(pooled) > POOL_CAP:\n            pooled = pooled[rng(43, self.seed, t).choice(len(pooled), POOL_CAP, replace=False)]\n        m = dict(P=LedoitWolf(assume_centered=True).fit(pooled).precision_, mub=self.Zref.mean(0), means=means)\n        m[\'tau\'] = conformal(self.score_z(self.Zcal, m), ALPHA); self.models[t] = m\n        return m\n\n    def score_z(self, Z, m):\n        if not len(Z):\n            return np.zeros(0)\n        return 0.5 * (maha_sq(Z, m[\'mub\'], m[\'P\']) - np.min([maha_sq(Z, mu, m[\'P\']) for mu in m[\'means\'].values()], axis=0))\n\n    def flags(self, X, t):\n        m = self.model(t); s = self.score_z(self.std(X), m); return s > m[\'tau\'], m[\'tau\']\n\n\ndef remove_rows(m, drop):\n    keep = np.setdiff1d(np.arange(m[\'size\']), drop)\n    for k in (\'x\', \'y\', \'ids\', \'z\', \'width\'):\n        m[k][:len(keep)] = m[k][keep]\n    m[\'size\'] = len(keep)\n\n\ndef audit(method, model, clients, slots, t, auditor, poison_ids, seed):\n    rows = []; bslot = slots.index(BENIGN)\n    for k, c in enumerate(clients):\n        m = c[\'replay\']\n        if m is None or m[\'size\'] == 0:\n            rows.append(dict(client=k, size=0)); continue\n        n = m[\'size\']; y, ids = m[\'y\'][:n], m[\'ids\'][:n]\n        is_p = np.isin(ids, poison_ids); ben = np.flatnonzero(y == BENIGN)\n        flag, tau = auditor.flags(m[\'x\'][:n][ben], t) if len(ben) else (np.zeros(0, bool), None)\n        budget = int(flag.sum())\n        if method == \'BRMA\':\n            drop = ben[flag]\n        elif method == \'RAND\':\n            drop = rng(44, seed, t, k).choice(ben, budget, replace=False) if budget else np.zeros(0, int)\n        elif method == \'LOSS\':\n            model.eval()\n            with torch.no_grad():\n                loss = F.cross_entropy(model(torch.from_numpy(m[\'x\'][:n][ben].copy())), torch.full((len(ben),), bslot), reduction=\'none\').numpy()\n            drop = ben[np.lexsort((ids[ben], -loss))[:budget]]\n        elif method == \'ORACLE\':\n            drop = np.flatnonzero(is_p)\n        else:\n            drop = np.zeros(0, int)\n        drop = np.asarray(drop, np.int64)\n        rows.append(dict(client=k, size=int(n), benign_labelled=int(len(ben)), poison_before=int(is_p.sum()), brma_flagged=budget,\n                         brma_flagged_poison=int(is_p[ben][flag].sum()) if len(ben) else 0, removed=int(len(drop)),\n                         removed_poison=int(is_p[drop].sum()), removed_genuine=int(len(drop) - is_p[drop].sum()), tau=tau))\n        remove_rows(m, drop)\n    return rows\n\n\ndef census(clients, poison_ids):\n    out = []\n    for c in clients:\n        m = c[\'replay\']\n        if m is None:\n            out.append(None); continue\n        n = m[\'size\']; out.append(dict(size=int(n), poison=int(np.isin(m[\'ids\'][:n], poison_ids).sum()), benign_labelled=int((m[\'y\'][:n] == BENIGN).sum())))\n    return out\n\n\n# ---------------------------------------------------------------- jobs\ndef key(seed, arm, method):\n    return f\'s{seed}-{arm}-{method}\'\n\n\ndef done(out, seed, arm, method):\n    return (Path(out) / \'runs\' / key(seed, arm, method) / \'result.json\').exists()\n\n\ndef save(out, k, payload):\n    f = Path(out) / \'runs\' / k; f.mkdir(parents=True, exist_ok=True)\n    tmp = f / \'result.json.tmp\'; tmp.write_text(json.dumps(payload)); tmp.replace(f / \'result.json\')\n\n\ndef cfg(job):\n    c = dict(job.get(\'cfg\') or {}); c.pop(\'features\', None)\n    if \'hidden\' in c:\n        c[\'hidden\'] = tuple(c[\'hidden\'])\n    return c\n\n\ndef worker(job):\n    setup()\n    folder, out, seed, arm = job[\'data\'], job[\'out\'], job[\'seed\'], job[\'arm\']\n    t0 = time.perf_counter()\n    data, attack = inputs(folder, seed, arm); d = data[0].x.shape[1]\n    pid = np.asarray(attack[\'poison_ids\'], np.uint64)\n    base = dict(seed=seed, arm=arm, arm_spec=ARMS[arm], attack=dict(attackers=attack[\'attackers\'], k=attack[\'k\'], dose=attack[\'dose\'], n_poison=len(pid)),\n                alpha=ALPHA, dataset=\'TON_IoT train_test_network\', target=CLASSES[TARGET])\n    if job[\'kind\'] == \'full\':\n        mth = job[\'method\']\n        s = Settings(**cfg(job), seed=seed, dose=ARMS[arm][1:], learner=mth if mth in (\'FT\', \'EI\') else \'DR\',\n                     admission=mth if mth in (\'DC\', \'DG\') else \'NONE\', aggregation=\'median\' if mth == \'MEDIAN\' else \'fedavg\')\n        clients = [client_state(s, i, d) for i in range(5)]\n        model, stages = run_stages(folder, None, clients, [], data, s, 1, 4)\n        save(out, key(seed, arm, mth), dict(base, method=mth, settings=asdict(s), stages=stages, memory=census(clients, pid), audits=[],\n                                            elapsed_seconds=time.perf_counter() - t0))\n        return [key(seed, arm, mth)]\n    s = Settings(**cfg(job), seed=seed, dose=ARMS[arm][1:])\n    clients = [client_state(s, i, d) for i in range(5)]; slots = []\n    model, prefix = run_stages(folder, None, clients, slots, data, s, 1, 2)\n    c2 = census(clients, pid); pre = time.perf_counter() - t0\n    auditor = Auditor(data, seed); keys = []\n    for mth in job[\'methods\']:\n        if done(out, seed, arm, mth):\n            continue\n        t1 = time.perf_counter()\n        bm, bc, bs = copy.deepcopy(model), copy.deepcopy(clients), list(slots)\n        audits = [dict(after_stage=2, clients=audit(mth, bm, bc, bs, 2, auditor, pid, seed))]\n        bm, st3 = run_stages(folder, bm, bc, bs, data, s, 3, 3)\n        audits.append(dict(after_stage=3, clients=audit(mth, bm, bc, bs, 3, auditor, pid, seed)))\n        bm, st4 = run_stages(folder, bm, bc, bs, data, s, 4, 4)\n        save(out, key(seed, arm, mth), dict(base, method=mth, settings=asdict(s), stages=prefix + st3 + st4, memory_after_stage2=c2,\n                                            memory=census(bc, pid), audits=audits, elapsed_seconds=pre + time.perf_counter() - t1, shared_prefix_seconds=pre))\n        keys.append(key(seed, arm, mth))\n    return keys\n\n\ndef build_jobs(folder, out, seeds, arms, branch, full, config=None):\n    jobs = []\n    for seed in seeds:\n        for arm in arms:\n            if branch and not all(done(out, seed, arm, m) for m in branch):\n                jobs.append(dict(kind=\'tree\', seed=seed, arm=arm, methods=list(branch), data=str(folder), out=str(out), cfg=config))\n            for m in full:\n                if not done(out, seed, arm, m):\n                    jobs.append(dict(kind=\'full\', seed=seed, arm=arm, method=m, data=str(folder), out=str(out), cfg=config))\n    jobs.sort(key=lambda j: (seeds.index(j[\'seed\']), j[\'kind\'] != \'tree\'))   # seed by seed: a time-out leaves whole seeds\n    return jobs\n\n\ndef run(a):\n    out = Path(a.out).resolve(); out.mkdir(parents=True, exist_ok=True)\n    config = json.loads(a.cfg) if a.cfg else None\n    (out / \'protocol.json\').write_text(json.dumps(dict(arms={k: ARMS[k] for k in a.arms}, branch_methods=a.branch, full_methods=a.full, alpha=ALPHA,\n        seeds=a.seeds, classes=CLASSES, arrival=ARRIVAL, target=CLASSES[TARGET], dirichlet=DIRICHLET,\n        learner=asdict(Settings(**cfg(dict(cfg=config))))), indent=1))\n    jobs = build_jobs(a.data, out, a.seeds, a.arms, a.branch, a.full, config); t0 = time.time(); fin = 0\n    print(f\'{len(jobs)} jobs pending\', flush=True)\n    deadline = t0 + a.hours * 3600; stopped = False\n    with futures.ProcessPoolExecutor(max_workers=a.workers) as pool:\n        pend = {pool.submit(worker, j): j for j in jobs}\n        for f in futures.as_completed(pend):\n            j = pend[f]\n            if not stopped and time.time() > deadline:\n                stopped = True; n = sum(x.cancel() for x in pend)\n                print(f\'time limit of {a.hours} h reached: {n} queued jobs cancelled, waiting for running jobs; re-run to resume\', flush=True)\n            if f.cancelled():\n                continue\n            try:\n                ks = f.result()\n            except Exception:\n                (out / f"failure-{j[\'kind\']}-s{j[\'seed\']}-{j[\'arm\']}-{j.get(\'method\', \'tree\')}.txt").write_text(traceback.format_exc())\n                print(\'FAILED\', j[\'kind\'], j[\'seed\'], j[\'arm\'], j.get(\'method\', \'\'), flush=True); continue\n            fin += 1; el = time.time() - t0\n            print(f"[{fin}/{len(jobs)}] {j[\'kind\']} s{j[\'seed\']} {j[\'arm\']} {j.get(\'method\', \'\')} -> {len(ks)} results | "\n                  f"{el / 60:.1f} min elapsed, ~{el / fin * (len(jobs) - fin) / 60:.0f} min left", flush=True)\n    print(\'complete\' if not build_jobs(a.data, out, a.seeds, a.arms, a.branch, a.full, config) else \'incomplete: re-run to resume\', flush=True)\n\n\n# ---------------------------------------------------------------- tuning (development seeds, dev split only)\nTUNE_SEEDS = (101, 102, 103)          # disjoint from the 12 evaluation seeds\nGRID = {}                             # learner options x feature set; selection is restricted to cost <= MAX_COST x base\nfor _f in (\'base\', \'ports\'):\n    for _n, _c in ((\'base\', {}), (\'mem2000\', dict(capacity=2000)), (\'mem2000_wide\', dict(capacity=2000, hidden=[256, 128])),\n                   (\'mem5000_wide\', dict(capacity=5000, hidden=[256, 128])),\n                   (\'ep2_mem2000_wide\', dict(epochs=2, capacity=2000, hidden=[256, 128])),\n                   (\'ep2_mem2000_wide_weighted\', dict(epochs=2, capacity=2000, hidden=[256, 128], class_weight=True)),\n                   (\'r24_mem2000\', dict(rounds=24, capacity=2000)),\n                   (\'ep2_mem5000_wide_lr2e-3\', dict(epochs=2, capacity=5000, hidden=[256, 128], learning_rate=.002))):\n        GRID[f\'{_f}__{_n}\'] = dict(_c, features=_f)\nMAX_COST = 3.0\n\n\ndef tune(a):\n    out = Path(a.out).resolve(); out.mkdir(parents=True, exist_ok=True)\n    jobs = []\n    for name, c in GRID.items():\n        for seed in a.seeds:\n            if not done(out / name, seed, \'C\', \'NONE\'):\n                folder = a.data_ports if c.get(\'features\') == \'ports\' else a.data\n                if c.get(\'features\') == \'ports\' and not folder:\n                    continue\n                jobs.append(dict(kind=\'full\', seed=seed, arm=\'C\', method=\'NONE\', data=str(folder), out=str(out / name), cfg=c))\n    print(f\'tuning: {len(GRID)} configurations x {len(a.seeds)} development seeds, {len(jobs)} jobs pending\', flush=True)\n    t0 = time.time(); fin = 0\n    with futures.ProcessPoolExecutor(max_workers=a.workers) as pool:\n        pend = {pool.submit(worker, j): j for j in jobs}\n        for f in futures.as_completed(pend):\n            j = pend[f]\n            try:\n                f.result()\n            except Exception:\n                (out / f"failure-{Path(j[\'out\']).name}-s{j[\'seed\']}.txt").write_text(traceback.format_exc()); print(\'FAILED\', j[\'out\'], j[\'seed\'], flush=True); continue\n            fin += 1; el = time.time() - t0\n            print(f"[{fin}/{len(jobs)}] {Path(j[\'out\']).name} s{j[\'seed\']} | {el / 60:.1f} min elapsed, ~{el / fin * (len(jobs) - fin) / 60:.0f} min left", flush=True)\n    rows = []\n    for name, c in GRID.items():\n        rs = [json.loads(p.read_text()) for p in sorted((out / name).glob(\'runs/*/result.json\'))]\n        if not rs:\n            continue\n        m = [result_row(r) for r in rs]\n        rows.append(dict(config=name, cfg=c, seeds=len(m), dev_macro_f1=float(np.mean([x[\'dev_macro_f1\'] for x in m])),\n                         dev_sd=float(np.std([x[\'dev_macro_f1\'] for x in m])), minutes=float(np.mean([r[\'elapsed_seconds\'] for r in rs]) / 60)))\n    rows.sort(key=lambda r: -r[\'dev_macro_f1\'])\n    base = next(r for r in rows if r[\'config\'] == \'base__base\')\n    for r in rows:\n        r[\'cost_vs_base\'] = r[\'minutes\'] / base[\'minutes\']\n    best = next(r for r in rows if r[\'cost_vs_base\'] <= MAX_COST)\n    (out / \'tuning.json\').write_text(json.dumps(dict(selection=f\'highest mean dev macro-F1 over development seeds 101-103 (clean, no defence) among settings costing <= {MAX_COST}x the base run time; test split not used\',\n                                                      best=best, table=rows), indent=1))\n    print()\n    print(\'configuration                          dev macro-F1 (%)   sd    min/run  cost\')\n    for r in rows:\n        print(f"{r[\'config\']:38s} {100 * r[\'dev_macro_f1\']:8.2f}      {100 * r[\'dev_sd\']:5.2f}  {r[\'minutes\']:6.1f}   {r[\'cost_vs_base\']:.1f}x")\n    print()\n    print(\'selected:\', best[\'config\'], json.dumps(best[\'cfg\']))\n\n\n# ---------------------------------------------------------------- metrics\ndef scores(M, eligible):\n    M = np.asarray(M, np.float64); f1 = {}\n    for c in eligible:\n        tp = M[c, c]; p = tp / M[:, c].sum() if M[:, c].sum() else 0.; r = tp / M[c].sum() if M[c].sum() else 0.\n        f1[c] = 2 * p * r / (p + r) if p + r else 0.\n    ben = M[BENIGN].sum()\n    return dict(f1=f1, macro_f1=float(np.mean([f1[c] for c in eligible])), accuracy=float(np.trace(M) / M.sum()),\n                benign_far=float((ben - M[BENIGN, BENIGN]) / ben) if ben else 0.,\n                target_recall=float(M[TARGET, TARGET] / M[TARGET].sum()) if M[TARGET].sum() else np.nan,\n                target_asr=float(M[TARGET, BENIGN] / M[TARGET].sum()) if M[TARGET].sum() else np.nan)\n\n\ndef result_row(r):\n    row = dict(seed=r[\'seed\'], arm=r[\'arm\'], method=r[\'method\'])\n    for split in (\'dev\', \'test\'):\n        st = [scores(s[split], [c for c in range(K) if ARRIVAL[c] <= s[\'stage\']]) for s in r[\'stages\']]\n        last = st[-1]\n        row.update({f\'{split}_macro_f1\': last[\'macro_f1\'], f\'{split}_accuracy\': last[\'accuracy\'], f\'{split}_far\': last[\'benign_far\'],\n                    f\'{split}_old_f1\': float(np.mean([last[\'f1\'][c] for c in range(K) if ARRIVAL[c] == 1])),\n                    f\'{split}_new_f1\': float(np.mean([last[\'f1\'][c] for c in range(K) if ARRIVAL[c] > 1])),\n                    f\'{split}_A\': float(np.mean([np.mean([st[t - 1][\'f1\'][c] for c in range(K) if ARRIVAL[c] == t and c != TARGET]) for t in (2, 3, 4)])),\n                    f\'{split}_e2_recall\': st[1][\'target_recall\'], f\'{split}_e3_asr\': st[2][\'target_asr\'], f\'{split}_e4_asr\': st[3][\'target_asr\']})\n        for c in range(K):\n            row[f\'{split}_f1_{CLASSES[c]}\'] = last[\'f1\'][c]\n    return row\n\n\ndef summarize(a):\n    import pandas as pd\n    out = Path(a.out)\n    rows = [result_row(json.loads(p.read_text())) for p in sorted(out.glob(\'runs/*/result.json\'))]\n    df = pd.DataFrame(rows)\n    clean = df[df.arm == \'C\'][[\'seed\', \'method\', \'dev_e3_asr\', \'dev_e4_asr\', \'test_e3_asr\', \'test_e4_asr\']]\n    clean = clean.rename(columns={c: \'clean_\' + c for c in clean.columns if c not in (\'seed\', \'method\')})\n    df = df.merge(clean, on=[\'seed\', \'method\'], how=\'left\')\n    for split in (\'dev\', \'test\'):\n        df[f\'{split}_H\'] = .5 * ((df[f\'{split}_e3_asr\'] - df[f\'clean_{split}_e3_asr\']) + (df[f\'{split}_e4_asr\'] - df[f\'clean_{split}_e4_asr\']))\n    df.to_csv(out / \'metrics.csv\', index=False)\n    pd.set_option(\'display.width\', 200)\n    print(f\'runs: {len(df)}, seeds: {sorted(df.seed.unique())}\')\n    print(\'\\nTest macro-F1 (%) after stage 4, all 10 classes  [Carillo et al. TNSM 2026, clean clients: F1 all = 74.49]\')\n    print((100 * df.pivot_table(index=\'method\', columns=\'arm\', values=\'test_macro_f1\')).round(2).to_string())\n    print(\'\\nTest accuracy (%) after stage 4\')\n    print((100 * df.pivot_table(index=\'method\', columns=\'arm\', values=\'test_accuracy\')).round(2).to_string())\n    print(\'\\nH (test): rise of DDoS-predicted-as-benign rate over the clean run, lower is better\')\n    print(df.pivot_table(index=\'method\', columns=\'arm\', values=\'test_H\').round(4).to_string())\n    c = df[(df.arm == \'C\')].groupby(\'method\')[[\'test_macro_f1\', \'test_old_f1\', \'test_new_f1\']].mean() * 100\n    print(\'\\nClean arm, test F1 (%): all / old (stage-1 classes) / new (later classes)   [Carillo: 74.49 / 79.05 / 68.10]\')\n    print(c.round(2).to_string())\n    best = df[(df.arm == \'C\') & (df.method == \'NONE\')].test_macro_f1.mean() * 100\n    print(f\'\\nOur clean replay learner (DR, no defence): {best:.2f}% vs Carillo 74.49%  ->  {"HIGHER" if best > CARILLO_F1_ALL else "LOWER"} by {abs(best - CARILLO_F1_ALL):.2f} points\')\n    return df\n\n\ndef main():\n    ap = argparse.ArgumentParser(); sub = ap.add_subparsers(dest=\'cmd\', required=True)\n    p = sub.add_parser(\'prepare\'); p.add_argument(\'--csv\', required=True); p.add_argument(\'--data\', required=True); p.add_argument(\'--ports\', action=\'store_true\')\n    r = sub.add_parser(\'run\'); r.add_argument(\'--data\', required=True); r.add_argument(\'--out\', required=True)\n    r.add_argument(\'--seeds\', type=int, nargs=\'+\', default=list(SEEDS)); r.add_argument(\'--workers\', type=int, default=4)\n    r.add_argument(\'--arms\', nargs=\'+\', default=list(ARMS)); r.add_argument(\'--branch\', nargs=\'*\', default=list(BRANCH_METHODS))\n    r.add_argument(\'--full\', nargs=\'*\', default=list(FULL_METHODS))\n    r.add_argument(\'--hours\', type=float, default=1000.); r.add_argument(\'--cfg\', default=\'\')\n    t = sub.add_parser(\'tune\'); t.add_argument(\'--data\', required=True); t.add_argument(\'--out\', required=True)\n    t.add_argument(\'--seeds\', type=int, nargs=\'+\', default=list(TUNE_SEEDS)); t.add_argument(\'--workers\', type=int, default=4)\n    t.add_argument(\'--data-ports\', default=\'\')\n    s = sub.add_parser(\'summarize\'); s.add_argument(\'--data\', required=True); s.add_argument(\'--out\', required=True)\n    a = ap.parse_args()\n    if a.cmd == \'prepare\':\n        print(json.dumps(prepare(a.csv, a.data, a.ports), indent=1))\n    elif a.cmd == \'run\':\n        run(a)\n    elif a.cmd == \'tune\':\n        tune(a)\n    else:\n        summarize(a)\n\n\nif __name__ == \'__main__\':\n    main()\n'
IMP_SRC = '"""BRMA improvement study on TON_IoT (DDoS weakness). Imports ton_study.py unchanged.\n\nMemory actions after stages 2 and 3 (same points as BRMA):\n  NONE, BRMA (Gaussian likelihood ratio, original), LOSS (BRMA\'s budget + loss ranking), ORACLE,\n  BRMA-M  model-margin score  m(x) = max_{c != benign} z_c(x) - z_benign(x)  of the current global model,\n          conformal threshold at level alpha on the stage-1 benign calibration half;\n  BRMA-G  Gaussian-mixture benign model (5 components) against tied Ledoit-Wolf attack Gaussians,\n          log-likelihood ratio, conformal threshold at level alpha;\n  BRMA-C  union of the original BRMA test and the model-margin test, each at level alpha/2\n          (Bonferroni: false-removal rate still <= alpha).\nPhase 1 (selection): development seeds 101-103, arms C/K3H/K5, DEV split. Rule (fixed in advance): among BRMA-M/G/C, keep variants whose\n  clean false-removal rate <= alpha + 0.01 and clean dev macro-F1 >= NONE - 1 point; choose the lowest mean dev H over K3H and K5.\nPhase 2 (evaluation): fresh seeds 201-212, arms C/K1/K3/K3H/K5, TEST split, all methods.\nPre-specified tests (one-sided paired Wilcoxon, Holm over four), S = selected variant:\n  T11 S < BRMA on H at K3H;  T12 S < BRMA on H at K5;  T13 S < NONE on H at K3H;  T14 clean macro-F1 of S non-inferior to NONE (margin 2 points).\nUsage:  python ton_improve.py run --data DATA_PORTS --out OUT --seeds 101 102 103 --arms C K3H K5\n"""\nfrom __future__ import annotations\nimport os\nfor _k in (\'OMP_NUM_THREADS\', \'MKL_NUM_THREADS\', \'OPENBLAS_NUM_THREADS\', \'NUMEXPR_NUM_THREADS\'):\n    os.environ[_k] = \'1\'\nimport argparse, copy, json, sys, time, traceback\nimport concurrent.futures as futures\nfrom dataclasses import asdict\nfrom pathlib import Path\nimport numpy as np\nimport torch\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport ton_study as T                                    # noqa: E402\n\nTON_CFG = dict(rounds=24, capacity=2000, features=\'ports\')   # selected earlier on development seeds (ton_study tune)\nBRANCH = (\'NONE\', \'BRMA\', \'LOSS\', \'ORACLE\', \'BRMA-M\', \'BRMA-G\', \'BRMA-C\')\nVARIANTS = (\'BRMA-M\', \'BRMA-G\', \'BRMA-C\')\nDEV_SEEDS, EVAL_SEEDS = (101, 102, 103), tuple(range(201, 213))\nGMM_K, GMM_FIT = 5, 20000\n\n\nclass Auditor2(T.Auditor):\n    """Original BRMA statistics plus the model-margin and mixture scores."""\n\n    def __init__(self, data, seed):\n        super().__init__(data, seed)\n        x1 = data[0].x\n        ben = T.rng(41, seed).permutation(np.flatnonzero(data[0].y == T.BENIGN))\n        self.Xcal = np.asarray(x1[ben[len(ben) // 2:]], np.float32)     # raw calibration half (same split as BRMA)\n        self.gmm = {}\n\n    # ---- model margin\n    @staticmethod\n    def margin(model, X, slots):\n        b = slots.index(T.BENIGN); model.eval()\n        with torch.no_grad():\n            z = model(torch.from_numpy(np.ascontiguousarray(X, dtype=np.float32))).numpy()\n        other = np.delete(z, b, axis=1)\n        return other.max(1) - z[:, b] if other.shape[1] else np.full(len(X), -np.inf)\n\n    # ---- Gaussian mixture for benign\n    def gmm_model(self, t):\n        if t in self.gmm:\n            return self.gmm[t]\n        from sklearn.mixture import GaussianMixture\n        m = self.model(t); d = self.Zref.shape[1]\n        Zr = self.Zref if len(self.Zref) <= GMM_FIT else self.Zref[T.rng(46, self.seed, t).choice(len(self.Zref), GMM_FIT, replace=False)]\n        g = GaussianMixture(GMM_K, covariance_type=\'full\', reg_covar=1e-2, random_state=int(T.rng(47, self.seed, t).integers(1 << 30)), max_iter=200).fit(Zr)\n        sign, logdetP = np.linalg.slogdet(m[\'P\'])\n        const = 0.5 * logdetP - 0.5 * d * np.log(2 * np.pi)\n        g_m = dict(g=g, const=const)\n        g_m[\'tau\'] = T.conformal(self.gmm_score_z(self.Zcal, m, g_m), T.ALPHA); self.gmm[t] = g_m\n        return g_m\n\n    @staticmethod\n    def gmm_score_z(Z, m, g_m):\n        if not len(Z):\n            return np.zeros(0)\n        att = np.max([g_m[\'const\'] - 0.5 * T.maha_sq(Z, mu, m[\'P\']) for mu in m[\'means\'].values()], axis=0)\n        return att - g_m[\'g\'].score_samples(Z)\n\n    def flags_variant(self, method, model, X, slots, t):\n        if method == \'BRMA-G\':\n            m = self.model(t); g_m = self.gmm_model(t); return self.gmm_score_z(self.std(X), m, g_m) > g_m[\'tau\']\n        tau_m = lambda a: T.conformal(self.margin(model, self.Xcal, slots), a)\n        if method == \'BRMA-M\':\n            return self.margin(model, X, slots) > tau_m(T.ALPHA)\n        m = self.model(t)                                                   # BRMA-C: union at alpha/2 each\n        tau_g = T.conformal(self.score_z(self.Zcal, m), T.ALPHA / 2)\n        return (self.score_z(self.std(X), m) > tau_g) | (self.margin(model, X, slots) > tau_m(T.ALPHA / 2))\n\n\ndef audit(method, model, clients, slots, t, auditor, poison_ids, seed):\n    if method not in VARIANTS:\n        return T.audit(method, model, clients, slots, t, auditor, poison_ids, seed)\n    rows = []\n    for k, c in enumerate(clients):\n        m = c[\'replay\']\n        if m is None or m[\'size\'] == 0:\n            rows.append(dict(client=k, size=0)); continue\n        n = m[\'size\']; ids = m[\'ids\'][:n]; is_p = np.isin(ids, poison_ids); ben = np.flatnonzero(m[\'y\'][:n] == T.BENIGN)\n        flag = auditor.flags_variant(method, model, m[\'x\'][:n][ben], slots, t) if len(ben) else np.zeros(0, bool)\n        drop = ben[flag].astype(np.int64)\n        rows.append(dict(client=k, size=int(n), benign_labelled=int(len(ben)), poison_before=int(is_p.sum()), brma_flagged=int(flag.sum()),\n                         brma_flagged_poison=int(is_p[ben][flag].sum()) if len(ben) else 0, removed=int(len(drop)),\n                         removed_poison=int(is_p[drop].sum()), removed_genuine=int(len(drop) - is_p[drop].sum())))\n        T.remove_rows(m, drop)\n    return rows\n\n\ndef worker(job):\n    T.setup()\n    folder, out, seed, arm = job[\'data\'], job[\'out\'], job[\'seed\'], job[\'arm\']\n    t0 = time.perf_counter()\n    data, attack = T.inputs(folder, seed, arm); d = data[0].x.shape[1]\n    pid = np.asarray(attack[\'poison_ids\'], np.uint64)\n    base = dict(seed=seed, arm=arm, arm_spec=T.ARMS[arm], attack=dict(attackers=attack[\'attackers\'], k=attack[\'k\'], dose=attack[\'dose\'], n_poison=len(pid)),\n                alpha=T.ALPHA, dataset=\'TON_IoT train_test_network (ports)\', target=T.CLASSES[T.TARGET], cfg=TON_CFG)\n    s = T.Settings(**T.cfg(dict(cfg=TON_CFG)), seed=seed, dose=T.ARMS[arm][1:])\n    clients = [T.client_state(s, i, d) for i in range(5)]; slots = []\n    model, prefix = T.run_stages(folder, None, clients, slots, data, s, 1, 2)\n    c2 = T.census(clients, pid); pre = time.perf_counter() - t0\n    auditor = Auditor2(data, seed); keys = []\n    for mth in BRANCH:\n        if T.done(out, seed, arm, mth):\n            continue\n        t1 = time.perf_counter()\n        bm, bc, bs = copy.deepcopy(model), copy.deepcopy(clients), list(slots)\n        audits = [dict(after_stage=2, clients=audit(mth, bm, bc, bs, 2, auditor, pid, seed))]\n        bm, st3 = T.run_stages(folder, bm, bc, bs, data, s, 3, 3)\n        audits.append(dict(after_stage=3, clients=audit(mth, bm, bc, bs, 3, auditor, pid, seed)))\n        bm, st4 = T.run_stages(folder, bm, bc, bs, data, s, 4, 4)\n        T.save(out, T.key(seed, arm, mth), dict(base, method=mth, settings=asdict(s), stages=prefix + st3 + st4, memory_after_stage2=c2,\n               memory=T.census(bc, pid), audits=audits, elapsed_seconds=pre + time.perf_counter() - t1, shared_prefix_seconds=pre))\n        keys.append(T.key(seed, arm, mth))\n    return keys\n\n\ndef run(a):\n    out = Path(a.out).resolve(); out.mkdir(parents=True, exist_ok=True)\n    (out / \'protocol.json\').write_text(json.dumps(dict(cfg=TON_CFG, branch=BRANCH, variants=VARIANTS, seeds=a.seeds, arms=a.arms, alpha=T.ALPHA,\n                                                         gmm_components=GMM_K, doc=__doc__), indent=1))\n    jobs = [dict(seed=sd, arm=ar, data=str(a.data), out=str(out)) for sd in a.seeds for ar in a.arms\n            if not all(T.done(out, sd, ar, m) for m in BRANCH)]\n    print(f\'{len(jobs)} jobs pending\', flush=True); t0 = time.time(); fin = 0\n    with futures.ProcessPoolExecutor(max_workers=a.workers) as pool:\n        pend = {pool.submit(worker, j): j for j in jobs}\n        for f in futures.as_completed(pend):\n            j = pend[f]\n            try:\n                ks = f.result()\n            except Exception:\n                (out / f"failure-s{j[\'seed\']}-{j[\'arm\']}.txt").write_text(traceback.format_exc()); print(\'FAILED\', j[\'seed\'], j[\'arm\'], flush=True); continue\n            fin += 1; el = time.time() - t0\n            print(f"[{fin}/{len(jobs)}] s{j[\'seed\']} {j[\'arm\']} -> {len(ks)} results | {el / 60:.1f} min elapsed, ~{el / fin * (len(jobs) - fin) / 60:.0f} min left", flush=True)\n\n\ndef table(out, split):\n    """Per-run metrics (macro-F1, H, false removal, detection) for one split."""\n    import pandas as pd\n    rows = []\n    for p in sorted(Path(out).glob(\'runs/*/result.json\')):\n        r = json.loads(p.read_text()); row = T.result_row(r)\n        a2 = [c for c in r[\'audits\'][0][\'clients\'] if c.get(\'size\')] if r[\'audits\'] else []\n        ben = sum(c.get(\'benign_labelled\', 0) for c in a2); pb = sum(c.get(\'poison_before\', 0) for c in a2)\n        rp = sum(c.get(\'removed_poison\', 0) for c in a2); rg = sum(c.get(\'removed_genuine\', 0) for c in a2)\n        rows.append(dict(seed=r[\'seed\'], arm=r[\'arm\'], method=r[\'method\'], macro_f1=row[f\'{split}_macro_f1\'], accuracy=row[f\'{split}_accuracy\'],\n                         e3=row[f\'{split}_e3_asr\'], e4=row[f\'{split}_e4_asr\'], poison=pb, removed_poison=rp,\n                         false_removal=rg / max(1, ben - pb), poison_end=sum(m[\'poison\'] for m in r[\'memory\'] if m)))\n    df = pd.DataFrame(rows); cl = df[df.arm == \'C\'].set_index([\'seed\', \'method\'])\n    df[\'H\'] = [.5 * ((x.e3 - cl.loc[(x.seed, x.method), \'e3\']) + (x.e4 - cl.loc[(x.seed, x.method), \'e4\'])) for x in df.itertuples()]\n    df[\'detection\'] = df.removed_poison / df.poison.replace(0, np.nan)\n    return df\n\n\ndef select(a):\n    df = table(a.out, \'dev\'); df.to_csv(Path(a.out) / \'dev-metrics.csv\', index=False)\n    import pandas as pd\n    pd.set_option(\'display.width\', 200)\n    g = df.groupby([\'method\', \'arm\'])[[\'H\', \'macro_f1\', \'false_removal\', \'detection\']].mean().unstack(\'arm\')\n    print(\'Phase 1 (development seeds, dev split): mean over seeds\'); print(g.round(4).to_string())\n    clean = df[df.arm == \'C\'].groupby(\'method\')[[\'macro_f1\', \'false_removal\']].mean()\n    hatt = df[df.arm.isin([\'K3H\', \'K5\'])].groupby(\'method\').H.mean()\n    ok = [v for v in VARIANTS if clean.loc[v, \'false_removal\'] <= T.ALPHA + 0.01 and clean.loc[v, \'macro_f1\'] >= clean.loc[\'NONE\', \'macro_f1\'] - 0.01]\n    best = min(ok, key=lambda v: hatt[v]) if ok else \'BRMA-C\'\n    print(\'\\neligible variants:\', ok, \'| mean dev H (K3H, K5):\', {v: round(float(hatt[v]), 4) for v in VARIANTS}, \'| BRMA:\', round(float(hatt[\'BRMA\']), 4))\n    print(\'SELECTED:\', best)\n    (Path(a.out) / \'selected.json\').write_text(json.dumps(dict(selected=best, eligible=ok, mean_dev_H={k: float(v) for k, v in hatt.items()},\n                                                              rule=\'eligible: clean false removal <= alpha+0.01 and clean dev macro-F1 >= NONE-1pt; choose lowest mean dev H over K3H,K5\')))\n\n\ndef evaluate_tests(a):\n    from scipy.stats import wilcoxon\n    import pandas as pd\n    S = json.loads(Path(a.sel).read_text())[\'selected\']\n    df = table(a.out, \'test\'); df.to_csv(Path(a.out) / \'test-metrics.csv\', index=False)\n    pd.set_option(\'display.width\', 200)\n    order = [\'NONE\', \'BRMA\', \'LOSS\', \'ORACLE\', \'BRMA-M\', \'BRMA-G\', \'BRMA-C\']; arms = [x for x in [\'C\', \'K1\', \'K3\', \'K3H\', \'K5\'] if x in set(df.arm)]\n    for col, sc in ((\'H\', 1), (\'macro_f1\', 100), (\'accuracy\', 100), (\'detection\', 100), (\'false_removal\', 100)):\n        print(f\'\\n{col} (test split, mean over seeds)\'); print((sc * df.pivot_table(index=\'method\', columns=\'arm\', values=col)).reindex(index=order, columns=arms).round(4).to_string())\n    def pair(arm, x, y, col):\n        u = df[(df.arm == arm) & (df.method == x)].set_index(\'seed\')[col]; v = df[(df.arm == arm) & (df.method == y)].set_index(\'seed\')[col].loc[u.index]\n        return u.to_numpy(), v.to_numpy()\n    res = []\n    for tid, arm, other in ((\'T11\', \'K3H\', \'BRMA\'), (\'T12\', \'K5\', \'BRMA\'), (\'T13\', \'K3H\', \'NONE\')):\n        u, v = pair(arm, S, other, \'H\'); d = u - v\n        res.append(dict(test=tid, hypothesis=f\'{S} < {other} on H at {arm}\', selected=u.mean(), other=v.mean(), wins=int((d < 0).sum()),\n                        p=float(wilcoxon(d, alternative=\'less\').pvalue) if (d != 0).any() else 1.))\n    u, v = pair(\'C\', S, \'NONE\', \'macro_f1\'); d = u - v + 0.02\n    res.append(dict(test=\'T14\', hypothesis=f\'{S} clean macro-F1 non-inferior to NONE (2 points)\', selected=u.mean(), other=v.mean(), wins=int((d > 0).sum()),\n                    p=float(wilcoxon(d, alternative=\'greater\').pvalue) if (d != 0).any() else 1.))\n    r = pd.DataFrame(res).sort_values(\'p\'); k = len(r); mx = 0; adj = []\n    for i, p in enumerate(r.p):\n        mx = max(mx, (k - i) * p); adj.append(min(1., mx))\n    r[\'p_holm\'] = adj; r = r.sort_index(); r[\'confirmed\'] = r.p_holm < .05\n    print(f\'\\nPre-specified tests (selected variant: {S})\'); print(r.round(5).to_string(index=False)); r.to_csv(Path(a.out) / \'tests.csv\', index=False)\n\n\ndef main():\n    ap = argparse.ArgumentParser(); sub = ap.add_subparsers(dest=\'cmd\', required=True)\n    r = sub.add_parser(\'run\'); r.add_argument(\'--data\', required=True); r.add_argument(\'--out\', required=True)\n    r.add_argument(\'--seeds\', type=int, nargs=\'+\', required=True); r.add_argument(\'--arms\', nargs=\'+\', required=True); r.add_argument(\'--workers\', type=int, default=4)\n    s = sub.add_parser(\'select\'); s.add_argument(\'--out\', required=True)\n    e = sub.add_parser(\'evaluate\'); e.add_argument(\'--out\', required=True); e.add_argument(\'--sel\', required=True)\n    a = ap.parse_args()\n    {\'run\': run, \'select\': select, \'evaluate\': evaluate_tests}[a.cmd](a)\n\n\nif __name__ == \'__main__\':\n    main()\n'
open(f"{W}/ton_study.py", "w").write(TON_SRC); open(f"{W}/ton_improve.py", "w").write(IMP_SRC)
print("ton_study.py", hashlib.sha256(TON_SRC.encode()).hexdigest()[:16], "| ton_improve.py", hashlib.sha256(IMP_SRC.encode()).hexdigest()[:16])
def run(cmd, logfile=f"{W}/ton-improve.log"):
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=W, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    with open(logfile, "a") as log:
        for line in p.stdout:
            print(line, end="", flush=True); log.write(line); log.flush()
    p.wait()
    if p.returncode: print("exit code", p.returncode)
    return p.returncode

In [ ]:
# 3. Prepare TON_IoT with destination-port features (same preparation as the tuned study; about 2 minutes)
DATA = f"{W}/ton-data-ports"
if not os.path.exists(f"{DATA}/ton.npz"):
    run([sys.executable, "ton_study.py", "prepare", "--csv", CSV, "--data", DATA, "--ports"])
rep = json.load(open(f"{DATA}/prepare-report.json")); print("rows", rep["rows"], "| features", rep["features"])

In [ ]:
# 4. Phase 1: development seeds 101-103, arms C/K3H/K5 -> select one variant on the DEV split (about 20-30 minutes)
OUT1 = f"{W}/ton-improve/phase1"; cpu = str(os.cpu_count())
run([sys.executable, "ton_improve.py", "run", "--data", DATA, "--out", OUT1, "--seeds", "101", "102", "103", "--arms", "C", "K3H", "K5", "--workers", cpu])
run([sys.executable, "ton_improve.py", "select", "--out", OUT1])
print(open(f"{OUT1}/selected.json").read())

In [ ]:
# 5. Phase 2: fresh seeds 201-212, arms C/K1/K3/K3H/K5, all methods (about 2-2.5 hours)
OUT2 = f"{W}/ton-improve/phase2"
run([sys.executable, "ton_improve.py", "run", "--data", DATA, "--out", OUT2, "--seeds", *map(str, range(201, 213)),
     "--arms", "C", "K1", "K3", "K3H", "K5", "--workers", cpu])
n_done = len(glob.glob(f"{OUT2}/runs/*/result.json")); print(f"results present: {n_done}/420", "(complete)" if n_done >= 420 else "(incomplete)")

In [ ]:
# 6. Results on the TEST split and the pre-specified tests T11-T14
run([sys.executable, "ton_improve.py", "evaluate", "--out", OUT2, "--sel", f"{OUT1}/selected.json"])
fails = glob.glob(f"{W}/ton-improve/**/failure-*.txt", recursive=True); print("failures:", len(fails))
for f in fails[:2]: print(open(f).read()[-1500:])

In [ ]:
# 7. Package
import zipfile
zp = f"{W}/ton-improve.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{W}/ton-improve/**/*", recursive=True) + [f"{W}/ton-improve.log", f"{DATA}/prepare-report.json"]:
        if os.path.isfile(p): z.write(p, os.path.relpath(p, W))
print("download:", zp, round(os.path.getsize(zp) / 1e6, 1), "MB")